<a href="https://colab.research.google.com/github/AresChen3/Fuquaquaqua/blob/main/Session7_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 7: Integrated Python case study

# Suspicious Transaction Detector

Imagine you’ve been hired as a data analyst at a credit card company investigating fraudulent activity. Your job is to get their basic suspicious transaction detector up and running.

The system should:
- Analyze historical transaction data
- Detect unusual or suspicious behavior
- Flag and output these transactions to a file called `flagged_transactions.csv`

However, the provided code is **buggy**! Your job is to fix the major bugs using your programming skills and (if needed) your prefered flavor of GenAI as an assistant.

**Primary issue to fix:**
- The `is_geo_jump` method currently uses a `city_coords` variable, but this is not defined.
- You must update the function to pull **geolocation data from the provided database** in order to compute distances.

There may be additional bugs to address.

Once fixed, the code should:
- Flag suspicious transactions
- Save them to `flagged_transactions.csv`
- Print a message confirming how many transactions were flagged

In [13]:
import sqlite3
import csv
import numpy as np
from datetime import datetime
from collections import defaultdict
from geopy.distance import geodesic

In [14]:
class Transaction:
    def __init__(self, row, user_history, db_conn, coord_cache):
        self.transaction_id = row["transaction_id"]
        self.user_id = row["user_id"]
        self.amount = row["amount"]
        self.timestamp = datetime.fromisoformat(row["timestamp"])
        self.merchant = row["merchant"]
        self.location = row["location"]
        self.user_history = user_history
        self.db_conn = db_conn
        self.coord_cache = coord_cache

    def is_high_value(self):
        amounts = self.user_history.get("amounts", [])
        if len(amounts) < 5:
            return False

        threshold = np.percentile(amounts, 99)
        return self.amount > threshold

    def get_coords(self, location):
        """Look up a 'City, ST' location in the cities table, with caching."""

        if location in self.coord_cache:
            return self.coord_cache[location]

        try:
            city, state = (
                part.strip()
                for part in location.rsplit(",", 1)
            )
        except (AttributeError, ValueError):
            self.coord_cache[location] = None
            return None

        result = self.db_conn.execute(
            """
            SELECT lat, lng
            FROM cities
            WHERE city = ? AND state_id = ?
            LIMIT 1
            """,
            (city, state),
        ).fetchone()

        if (
            result is None
            or result[0] is None
            or result[1] is None
        ):
            coords = None
        else:
            coords = (
                float(result[0]),
                float(result[1]),
            )

        self.coord_cache[location] = coords
        return coords

    def is_geo_jump(self):
        locs = self.user_history.get("locations", [])
        times = self.user_history.get("timestamps", [])

        if not locs or not times:
            return False

        prev_location = locs[-1]
        prev_time = times[-1]

        if prev_location == self.location:
            return False

        coords1 = self.get_coords(prev_location)
        coords2 = self.get_coords(self.location)

        if coords1 is None or coords2 is None:
            return False

        distance_km = geodesic(
            coords1,
            coords2,
        ).kilometers

        time_diff_hr = (
            self.timestamp - prev_time
        ).total_seconds() / 3600

        return (
            distance_km > 400
            and time_diff_hr < distance_km / 500
        )

    def is_unusual_vendor(self):
        merchants = self.user_history.get(
            "merchants",
            set(),
        )

        return (
            len(merchants) >= 5
            and self.merchant not in merchants
        )

    def is_suspicious(self):
        reasons = []

        if self.is_high_value():
            reasons.append("High value")

        if self.is_geo_jump():
            reasons.append("Unrealistic geo jump")

        if self.is_unusual_vendor():
            reasons.append("New vendor")

        return reasons

In [15]:
def run_pipeline(
    db_path="fraud.db",
    cutoff="2024-07-15 00:00:00",
    output_file="flagged_transactions.csv",
):
    cutoff_dt = datetime.fromisoformat(cutoff)

    user_history = defaultdict(
        lambda: {
            "amounts": [],
            "locations": [],
            "timestamps": [],
            "merchants": set(),
        }
    )

    coord_cache = {}
    flagged = []

    # Keep the connection open while Transaction objects
    # query city coordinates.
    with sqlite3.connect(db_path) as conn:
        cursor = conn.execute(
            """
            SELECT *
            FROM transactions
            ORDER BY timestamp
            """
        )

        col_names = [
            desc[0]
            for desc in cursor.description
        ]

        for row in cursor:
            record = dict(
                zip(col_names, row)
            )

            txn_time = datetime.fromisoformat(
                record["timestamp"]
            )

            user_id = record["user_id"]
            profile = user_history[user_id]

            txn = Transaction(
                record,
                profile,
                conn,
                coord_cache,
            )

            if txn_time < cutoff_dt:
                profile["amounts"].append(
                    txn.amount
                )

                profile["locations"].append(
                    txn.location
                )

                profile["timestamps"].append(
                    txn.timestamp
                )

                profile["merchants"].add(
                    txn.merchant
                )

            else:
                reasons = txn.is_suspicious()

                if reasons:
                    flagged.append(
                        [
                            txn.transaction_id,
                            txn.user_id,
                            txn.timestamp.isoformat(),
                            "; ".join(reasons),
                        ]
                    )

    with open(
        output_file,
        "w",
        newline="",
        encoding="utf-8",
    ) as f:
        writer = csv.writer(f)

        writer.writerow(
            [
                "transaction_id",
                "user_id",
                "timestamp",
                "reasons",
            ]
        )

        writer.writerows(flagged)

    print(
        f"Flagged {len(flagged)} transactions. "
        f"Output saved to {output_file}."
    )

In [16]:
run_pipeline()

Flagged 6867 transactions. Output saved to flagged_transactions.csv.
